<a href="https://colab.research.google.com/github/jkrish006/DSA_assessment/blob/main/Data_Acquisition_casestudy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime

In [15]:
def extract():
    population = pd.read_csv("/content/population.csv")
    print("Population Dataset")
    print(population.head())
    print(population.shape)
    print(population.dtypes)

    gdp = pd.read_excel("/content/gdp.xlsx")
    print("\nGDP Dataset")
    print(gdp.columns)
    print(gdp.dtypes)

    internet = pd.read_json("/content/internet_users.json")
    internet=pd.DataFrame(internet)

    literacy = pd.read_xml("/content/literacy_rate.xml")
    literacy=pd.DataFrame(literacy)

    return population, gdp, internet, literacy

population, gdp, internet, literacy= extract()

Population Dataset
         country  population
0  United States   331002651
1          China  1411778724
2          India  1380004385
3          Japan   125836021
4        Germany    83240525
(25, 2)
country       object
population     int64
dtype: object

GDP Dataset
Index(['Country', 'GDP'], dtype='object')
Country    object
GDP         int64
dtype: object


In [36]:
def transform(population, gdp, internet, literacy):

    print("Population Dataset")

    print("Number of Rows:", population.shape[0])
    print("Number of Columns:", population.shape[1])
    print("Column Names:", population.columns)
    print("Data Types:")
    print(population.dtypes)
    print("Missing Values:")
    print(population.isnull().sum())
    print("Duplicate Records:", population.duplicated().sum())

    print("\nGDP Dataset")
    print("Number of Rows:", gdp.shape[0])
    print("Number of Columns:", gdp.shape[1])
    print("Column Names:", gdp.columns)
    print("Data Types:")
    print(gdp.dtypes)
    print("Missing Values:")
    print(gdp.isnull().sum())
    print("Duplicate Records:", gdp.duplicated().sum())

    print("\nInternet Users Dataset")
    print("Number of Rows:", internet.shape[0])
    print("Number of Columns:", internet.shape[1])
    print("Column Names:", internet.columns)
    print("Data Types:")
    print(internet.dtypes)
    print("Missing Values:")
    print(internet.isnull().sum())
    print("Duplicate Records:", internet.duplicated().sum())

    print("\nLiteracy Rate Dataset")
    print("Number of Rows:", literacy.shape[0])
    print("Number of Columns:", literacy.shape[1])
    print("Column Names:", literacy.columns)
    print("Data Types:")
    print(literacy.dtypes)
    print("Missing Values:")
    print(literacy.isnull().sum())
    print("Duplicate Records:", literacy.duplicated().sum())

    return population, gdp, internet, literacy
population, gdp, internet, literacy = transform(population, gdp, internet, literacy)

Population Dataset
Number of Rows: 25
Number of Columns: 2
Column Names: Index(['country', 'population'], dtype='object')
Data Types:
country       object
population     int64
dtype: object
Missing Values:
country       0
population    0
dtype: int64
Duplicate Records: 0

GDP Dataset
Number of Rows: 25
Number of Columns: 2
Column Names: Index(['country', 'GDP'], dtype='object')
Data Types:
country    object
GDP         int64
dtype: object
Missing Values:
country    0
GDP        0
dtype: int64
Duplicate Records: 0

Internet Users Dataset
Number of Rows: 25
Number of Columns: 2
Column Names: Index(['country', 'internet_users'], dtype='object')
Data Types:
country           object
internet_users     int64
dtype: object
Missing Values:
country           0
internet_users    0
dtype: int64
Duplicate Records: 0

Literacy Rate Dataset
Number of Rows: 25
Number of Columns: 2
Column Names: Index(['country', 'literacy_rate'], dtype='object')
Data Types:
country           object
literacy_rate    f

In [40]:
def transform(population, gdp, internet, literacy):


    gdp.rename(columns={"Country":"country"}, inplace=True)
    literacy.rename(columns={"name":"country"}, inplace=True)


    population["population"] = pd.to_numeric(population["population"])
    gdp["GDP"] = pd.to_numeric(gdp["GDP"])
    internet["internet_users"] = pd.to_numeric(internet["internet_users"])
    literacy["literacy_rate"] = pd.to_numeric(literacy["literacy_rate"])


    merged = pd.merge(population, gdp, on="country")
    merged = pd.merge(merged, internet, on="country")
    merged = pd.merge(merged, literacy, on="country")


    merged["Internet Penetration (%)"] = round(
        (merged["internet_users"] / merged["population"]) * 100, 2
    )

    merged["GDP Per Capita"] = round(
        merged["GDP"] / merged["population"], 2
    )

    print(merged.head())

    return merged
merged = transform(population, gdp, internet, literacy)

         country  population       GDP  internet_users  literacy_rate  \
0  United States   331002651  30500000       312000000           99.0   
1          China  1411778724  19230000      1080000000           97.2   
2          India  1380004385   4187000       850000000           77.7   
3          Japan   125836021   4180000       118000000           99.0   
4        Germany    83240525   4744000        79000000           99.0   

   Internet Penetration (%)  GDP Per Capita  
0                     94.26            0.09  
1                     76.50            0.01  
2                     61.59            0.00  
3                     93.77            0.03  
4                     94.91            0.06  


In [42]:
import sqlite3

def load(merged):

    # Task 10 - Create SQLite Database
    conn = sqlite3.connect("country_statistics.db")

    merged.to_sql(
        "country_statistics",
        conn,
        if_exists="replace",
        index=False
    )

    print("Data loaded successfully!")

    # Task 11 - Verify Data
    query = "SELECT * FROM country_statistics LIMIT 5"

    result = pd.read_sql(query, conn)

    print("\nFirst Five Records")
    print(result)

    conn.close()
load(merged)

Data loaded successfully!

First Five Records
         country  population       GDP  internet_users  literacy_rate  \
0  United States   331002651  30500000       312000000           99.0   
1          China  1411778724  19230000      1080000000           97.2   
2          India  1380004385   4187000       850000000           77.7   
3          Japan   125836021   4180000       118000000           99.0   
4        Germany    83240525   4744000        79000000           99.0   

   Internet Penetration (%)  GDP Per Capita  
0                     94.26            0.09  
1                     76.50            0.01  
2                     61.59            0.00  
3                     93.77            0.03  
4                     94.91            0.06  


In [45]:
print("Top 5 Countries by GDP")
print(merged.nlargest(5, "GDP")[["country", "GDP"]])

Top 5 Countries by GDP
         country       GDP
0  United States  30500000
1          China  19230000
4        Germany   4744000
2          India   4187000
3          Japan   4180000


In [43]:
print("Top 5 Countries by Population")
print(merged.nlargest(5, "population")[["country", "population"]])

Top 5 Countries by Population
          country  population
1           China  1411778724
2           India  1380004385
0   United States   331002651
15      Indonesia   273523615
10         Brazil   212559417


In [44]:
print("Average Literacy Rate")
print(merged["literacy_rate"].mean())

Average Literacy Rate
97.248


In [46]:
print("Countries with Literacy Rate > 90%")
print(merged[merged["literacy_rate"] > 90][["country", "literacy_rate"]])

Countries with Literacy Rate > 90%
           country  literacy_rate
0    United States           99.0
1            China           97.2
3            Japan           99.0
4          Germany           99.0
5   United Kingdom           99.0
6           France           99.0
7            Italy           99.0
8           Canada           99.0
9        Australia           99.0
10          Brazil           93.2
11          Russia           99.7
12     South Korea           98.8
13           Spain           98.6
14          Mexico           95.4
15       Indonesia           96.0
16    Saudi Arabia           97.3
17          Turkey           96.7
18    South Africa           95.0
19       Argentina           99.1
20     Netherlands           99.0
21     Switzerland           99.0
22          Sweden           99.0
23          Norway          100.0
24       Singapore           97.5


In [47]:
print("Countries with Internet Penetration > 70%")
print(merged[merged["Internet Penetration (%)"] > 70][["country", "Internet Penetration (%)"]])

Countries with Internet Penetration > 70%
           country  Internet Penetration (%)
0    United States                     94.26
1            China                     76.50
3            Japan                     93.77
4          Germany                     94.91
5   United Kingdom                     97.22
6           France                     94.98
7            Italy                     92.62
8           Canada                     94.72
9        Australia                     93.43
10          Brazil                     85.15
11          Russia                     85.65
12     South Korea                     96.56
13           Spain                     94.11
14          Mexico                     79.11
15       Indonesia                     77.51
16    Saudi Arabia                     97.66
17          Turkey                     81.81
18    South Africa                     72.50
19       Argentina                     90.72
20     Netherlands                     97.47
21     Switze

In [48]:
print("Top 5 Countries by GDP Per Capita")
print(merged.nlargest(5, "GDP Per Capita")[["country", "GDP Per Capita"]])

Top 5 Countries by GDP Per Capita
          country  GDP Per Capita
21    Switzerland            0.11
23         Norway            0.11
0   United States            0.09
24      Singapore            0.09
9       Australia            0.07


In [49]:
print("Total Population")
print(merged["population"].sum())

Total Population
4720039725


In [50]:
print("Average Internet Penetration")
print(merged["Internet Penetration (%)"].mean())

Average Internet Penetration
89.6596


In [53]:
print("Country with Highest Internet Users")
print(merged.loc[merged["internet_users"].idxmax(), ["country", "internet_users"]])

Country with Highest Internet Users
country                China
internet_users    1080000000
Name: 1, dtype: object


In [52]:
print("Correlation")
print(merged["literacy_rate"].corr(merged["Internet Penetration (%)"]))

Correlation
0.77279756813985


In [57]:
import sqlite3

conn = sqlite3.connect("country_statistics.db")
cursor = conn.cursor()

query = """
select country, gdp
from country_statistics order by gdp desc limit 5;"""

print(pd.read_sql(query, conn))

         country       GDP
0  United States  30500000
1          China  19230000
2        Germany   4744000
3          India   4187000
4          Japan   4180000


In [58]:
query = """
select AVG(literacy_rate) as Average_Literacy_Rate
from country_statistics;"""

print(pd.read_sql(query, conn))

   Average_Literacy_Rate
0                 97.248


In [59]:
query = """
select country, literacy_rate from country_statistics where literacy_rate > 90;"""

print(pd.read_sql(query, conn))

           country  literacy_rate
0    United States           99.0
1            China           97.2
2            Japan           99.0
3          Germany           99.0
4   United Kingdom           99.0
5           France           99.0
6            Italy           99.0
7           Canada           99.0
8        Australia           99.0
9           Brazil           93.2
10          Russia           99.7
11     South Korea           98.8
12           Spain           98.6
13          Mexico           95.4
14       Indonesia           96.0
15    Saudi Arabia           97.3
16          Turkey           96.7
17    South Africa           95.0
18       Argentina           99.1
19     Netherlands           99.0
20     Switzerland           99.0
21          Sweden           99.0
22          Norway          100.0
23       Singapore           97.5


In [62]:
query = """
select country, [Internet Penetration (%)] from country_statistics order BY [Internet Penetration (%)] desc limit 5;"""
print(pd.read_sql(query, conn))

        country  Internet Penetration (%)
0   Switzerland                     98.21
1        Norway                     97.76
2  Saudi Arabia                     97.66
3        Sweden                     97.55
4   Netherlands                     97.47


In [63]:
query = """
select SUM(population) as Total_Population from country_statistics;"""

print(pd.read_sql(query, conn))

   Total_Population
0        4720039725


In [64]:
query = """
select country, [GDP Per Capita] from country_statistics order by [GDP Per Capita] desc limit 1;"""

print(pd.read_sql(query, conn))

       country  GDP Per Capita
0  Switzerland            0.11


In [65]:
merged.to_csv("country_statistics.csv", index=False)

print("CSV Exported Successfully")

CSV Exported Successfully


In [67]:
query = """
select country, [GDP Per Capita] from country_statistics order by [GDP Per Capita] desc limit 10;
"""

result = pd.read_sql(query, conn)
print(result)

          country  GDP Per Capita
0     Switzerland            0.11
1          Norway            0.11
2   United States            0.09
3       Singapore            0.09
4       Australia            0.07
5     Netherlands            0.07
6         Germany            0.06
7  United Kingdom            0.06
8          Canada            0.06
9          Sweden            0.06


summary

In this ETL project, data was extracted from four different file formats: **population.csv**, **gdp.xlsx**, **internet_users.json**, and **literacy_rate.xml**.

The datasets were explored by checking the number of rows and columns, column names, data types, missing values, and duplicate records. During the transformation phase, column names were standardized to use a common **country** column, the required columns were converted to numeric data types, and all four datasets were merged into a single DataFrame. Two new calculated columns, **Internet Penetration (%)** and **GDP Per Capita**, were created to enhance the analysis.

 The final merged dataset was then loaded into a SQLite database named **country_statistics.db** using the pandas `to_sql()` function, and the data was verified by retrieving the first five records from the database. Finally, the dataset was analyzed using both Pandas and SQL to identify the top countries by GDP and population, calculate average literacy rate and internet penetration, and gain insights into the relationship between literacy, internet usage, and economic performance.
